# Experiment

In [1]:
import pyterrier as pt
import pandas as pd
import os
from pathlib import Path

In [2]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

## Index Creation/Loading

In [3]:
index_dir_path = Path("index/trec_covid_index").resolve()
index_dir = str(index_dir_path)
index_properties_file = os.path.join(index_dir, "data.properties")

In [4]:
os.makedirs(index_dir, exist_ok=True)

In [5]:
def trec_covid_corpus_generator():
    for doc in dataset.get_corpus_iter():
        title = str(doc.get('title', ''))
        abstract = str(doc.get('abstract', ''))
        
        yield {
            'docno': doc['docno'],
            'text': title + " " + abstract
        }

In [6]:
if not pt.started():
    pt.init()

C:\Users\zosia\AppData\Local\Temp\ipykernel_22868\2550260977.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
C:\Users\zosia\AppData\Local\Temp\ipykernel_22868\2550260977.py:2: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [7]:
if os.path.exists(index_properties_file):
    print(f"Index found at {index_dir}. Loading it...")
    index_ref = pt.IndexRef.of(index_properties_file)
else:
    print(f"Index not found at {index_dir}. Building it now (this may take a few minutes)...")
    
    indexer = pt.IterDictIndexer(index_dir, blocks=True, meta={'docno': 50})
    index_ref = indexer.index(trec_covid_corpus_generator())
    print("Index built successfully.")

Index not found at C:\Users\zosia\Desktop\Studies\Information Retrieval\IR_Project\src\index\trec_covid_index. Building it now (this may take a few minutes)...


cord19/trec-covid documents:   0%|          | 0/192509 [00:00<?, ?it/s]

18:27:16.573 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
18:29:38.126 [main] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
For MetaIndex, to suppress, set metaindex.compressed.reverse.allow.duplicates=true
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(Indexer.java:377)
18:29:49.940 [main] WA

## Different Query Verbosity Levels

In [8]:
topics_data = []
for query in dataset.irds_ref().queries_iter():
    topics_data.append({
        'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })
df_all_topics = pd.DataFrame(topics_data)

In [9]:
# TITLE Queries
topics_title = df_all_topics[['qid', 'title']].copy()
topics_title = topics_title.rename(columns={'title': 'query'})

# DESCRIPTION Queries
topics_desc = df_all_topics[['qid', 'description']].copy()
topics_desc = topics_desc.rename(columns={'description': 'query'})

# NARRATIVE Queries
topics_narr = df_all_topics[['qid', 'narrative']].copy()
topics_narr = topics_narr.rename(columns={'narrative': 'query'})

## Models

In [10]:
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25")
ql_dir = pt.terrier.Retriever(index_ref, wmodel="DirichletLM")
ql_jm = pt.terrier.Retriever(index_ref, wmodel="Hiemstra_LM")
rm3_pipe = bm25 >> pt.rewrite.RM3(index_ref) >> bm25

In [11]:
models = [bm25, ql_dir, ql_jm, rm3_pipe]
model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3"]

## Experiment

In [12]:
eval_metrics = [
    "ndcg_cut_10",
    "P_5",
    "recip_rank",
    "recall_100",
    "map"
]
query_variants = {
    "Title": topics_title,
    "Description": topics_desc,
    "Narrative": topics_narr
}

In [13]:
for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,0.000784,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,0.000003,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,0.695299,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01




Results for DESCRIPTION Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493




Results for NARRATIVE Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629


## Experiment with Dense Retrieval (DPR)

In [16]:
!pip install -q torch torchvision torchaudio transformers
!pip install -q faiss-cpu 
!pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_dr.git


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import pyterrier_dr as ptdr
import os
from pathlib import Path

dense_index_dir = str(Path("index/trec_covid_dense").resolve())

if os.path.exists(os.path.join(dense_index_dir, "pt_meta.json")):
    print("Found downloaded dense index! Loading...")
    dpr_model = ptdr.SBertBiEncoder('sentence-transformers/all-MiniLM-L6-v2')
    dense_index = ptdr.FlexIndex(dense_index_dir)
    dpr_pipe = dpr_model >> dense_index
    
    print("DPR Pipeline ready!")
else:
    print(f"Could not find the dense index at {dense_index_dir}. Please check where you extracted it!")

Found downloaded dense index! Loading...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR Pipeline ready!


In [18]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "DPR (all-MiniLM)"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,7.844063e-04,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,2.983346e-06,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,6.952988e-01,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01
4,DPR (all-MiniLM),0.075150,0.569682,0.336,0.050606,0.300213,3.0,47.0,2.424788e-09,7.0,...,0.000895,3.0,34.0,1.087754e-08,5.0,43.0,1.140979e-08,6.0,43.0,2.638420e-09




Results for DESCRIPTION Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493
4,DPR (all-MiniLM),0.119925,0.680775,0.528,0.075400,0.448695,6.0,44.0,2.219091e-09,5.0,...,0.004876,9.0,28.0,0.001194,8.0,41.0,1.290954e-06,12.0,38.0,0.000475




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,DPR (all-MiniLM),0.111516,0.543542,0.424,0.069849,0.367265,14.0,36.0,2.399987e-03,9.0,...,0.023483,9.0,28.0,0.003070,13.0,36.0,0.004678,11.0,35.0,0.000372


# BioDPR

In [1]:
!pip install -q --upgrade torch torchvision torchaudio


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pyterrier_dr as ptdr
import os
from pathlib import Path

biodpr_index_dir = str(Path("index/trec_covid_biodpr_complete").resolve())

if os.path.exists(os.path.join(biodpr_index_dir, "pt_meta.json")):
    print("Found BioDPR index! Loading...")
    
    biodpr_model = ptdr.SBertBiEncoder("pritamdeka/S-PubMedBert-MS-MARCO")
    biodpr_index = ptdr.FlexIndex(biodpr_index_dir)
    biodpr_pipe = biodpr_model >> biodpr_index
    
    print("BioDPR Pipeline ready!")
else:
    print(f"Could not find the index at {biodpr_index_dir}. Please make sure you extracted the folder correctly!")

Found BioDPR index! Loading...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BioDPR Pipeline ready!


In [15]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, biodpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "BioDPR"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,0.000784,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,0.000003,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,0.695299,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01
4,BioDPR,0.100963,0.553502,0.420,0.068487,0.348634,8.0,42.0,3.971513e-07,4.0,...,0.000041,7.0,32.0,0.000003,5.0,44.0,1.265811e-06,9.0,40.0,1.406583e-07




Results for DESCRIPTION Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493
4,BioDPR,0.168095,0.808270,0.652,0.096608,0.597648,14.0,36.0,7.343930e-04,6.0,...,0.204821,18.0,24.0,0.194325,15.0,32.0,3.710228e-02,24.0,26.0,0.544790




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,BioDPR,0.140223,0.786781,0.600,0.089901,0.551742,20.0,30.0,2.476971e-01,12.0,...,0.245488,17.0,18.0,0.878472,21.0,28.0,0.903510,25.0,23.0,0.409129


# Hybrid Approach

In [16]:
import pyterrier as pt

def normalize_scores(res):
    res = res.copy()
    
    grouped = res.groupby('qid')['score']
    min_score = grouped.transform('min')
    max_score = grouped.transform('max')
    
    res['score'] = (res['score'] - min_score) / (max_score - min_score).replace(0, 1e-5)
    
    return res

norm_transformer = pt.apply.generic(normalize_scores)

bm25_norm = bm25 >> norm_transformer
biodpr_norm = biodpr_pipe >> norm_transformer

hybrid_pipe = bm25_norm + biodpr_norm

all_models = [bm25, ql_dir, ql_jm, rm3_pipe, biodpr_pipe, hybrid_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "BioDPR", "Hybrid (BM25 + BioDPR)"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


C:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #5: Hybrid (BM25 + BioDPR) (<pyterrier._ops.Sum object at 0x000001EFDD4AC170>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,0.000784,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,0.000003,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,0.695299,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01
4,BioDPR,0.100963,0.553502,0.420,0.068487,0.348634,8.0,42.0,3.971513e-07,4.0,...,0.000041,7.0,32.0,0.000003,5.0,44.0,1.265811e-06,9.0,40.0,1.406583e-07
5,Hybrid (BM25 + BioDPR),0.227331,0.783354,0.652,0.110623,0.570775,33.0,17.0,1.908420e-03,7.0,...,0.576923,15.0,18.0,0.645718,22.0,23.0,1.013263e-01,23.0,23.0,3.401945e-01


...



Results for NARRATIVE Queries:


C:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #5: Hybrid (BM25 + BioDPR) (<pyterrier._ops.Sum object at 0x000001EFDD4AC170>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,BioDPR,0.140223,0.786781,0.600,0.089901,0.551742,20.0,30.0,2.476971e-01,12.0,...,0.245488,17.0,18.0,0.878472,21.0,28.0,0.903510,25.0,23.0,0.409129
5,Hybrid (BM25 + BioDPR),0.211106,0.838741,0.716,0.106440,0.652635,42.0,8.0,1.575750e-07,13.0,...,0.030661,20.0,7.0,0.008309,35.0,13.0,0.000163,36.0,11.0,0.000017
